# 2023년 제7차 근로환경조사(KWCS) 분석

## 단계: 01. 데이터 전처리 (Data Preprocessing)
- 목표: 5만 명 규모의 설문 원시자료를 분석 가능한 형태로 정제
- 데이터: 산업안전보건연구원 제7차 근로환경조사(2023), 50,195명 × 439개 변수
- 이전 프로젝트와의 차이: 지역 단위 집계 데이터 → 개인 단위 표본조사 마이크로데이터

### 1.1 원시자료 로드
- 확장자는 .csv이나 실제 구분자는 탭(\t)이며, 각 행 전체가 큰따옴표로 감싸져 있고 줄 끝에 쉼표 3개가 붙어 있음
- `quoting=csv.QUOTE_NONE`으로 따옴표를 무시하고 읽은 뒤, 첫/끝 컬럼의 잔여 문자를 제거
- `dtype=str`로 읽는 이유: 결측코드가 섞인 상태에서 숫자로 읽으면 컬럼별 타입 추론이 어긋남
- 공백 문자(' ')는 설문 미해당(skip pattern)을 의미하므로 결측(NA)으로 변환
- 원본 파일은 엑셀로 열지 않음. 엑셀은 저장 시 구분자를 쉼표로 바꾸고 유효숫자를 표시값 기준으로 잘라냄

In [4]:
import pandas as pd
import csv

filepath = r'C:\data\2023년 제7차 근로환경조사 원시자료.csv'

# 구분자는 탭, 인코딩은 cp949, 행 전체를 감싼 따옴표는 무시(QUOTE_NONE)
df = pd.read_csv(filepath, sep='\t', encoding='cp949',
                 quoting=csv.QUOTE_NONE, dtype=str)

# 첫/끝 컬럼에 붙은 따옴표와 꼬리 쉼표 제거
df.columns = [c.strip().strip('"').rstrip(',').strip('"') for c in df.columns]
first, last = df.columns[0], df.columns[-1]
df[first] = df[first].str.lstrip('"')
df[last]  = df[last].str.replace(r'",{0,3}$', '', regex=True)
df = df.replace(' ', pd.NA)

print(f'행 : {df.shape[0]}, 열 : {df.shape[1]}')   # 50195, 439
print(df.columns[:8].tolist())

행 : 50195, 열 : 439
['id', 'wt1', 'wt2', 'wt3', 'area', 'gender', 'year', 'age']


### 1.2 파일 무결성 검증
- 5만 행 규모에서는 데이터 오염을 눈으로 확인할 수 없으므로 수치로 대조
- 코드북에 변수별 빈도가 기재되어 있어 이를 정답지로 사용
- 검증 항목: emp_type, gender, satisfaction의 값별 빈도 및 wt3의 분포
- 하나라도 어긋나면 파일 손상으로 판단하고 재다운로드

In [5]:
# 코드북 기재값과 대조 — 하나라도 어긋나면 파일이 오염된 것
expected = {
    'emp_type'    : {'1': 12867, '2': 3138, '3': 30150, '4': 4040},
    'gender'      : {'1': 23678, '2': 26517},
    'satisfaction': {'1': 2379, '2': 37688, '3': 8458,
                     '4': 1372, '8': 275, '9': 23},
}

for var, exp in expected.items():
    got = df[var].value_counts().sort_index().to_dict()
    print(f'{var:13s} {"통과" if got == exp else "불일치"}')

w = pd.to_numeric(df['wt3'], errors='coerce')
print(f'wt3 평균 {w.mean():.4f} (기대 1.0000), 최대 {w.max():.4f} (기대 20.7537)')

emp_type      통과
gender        통과
satisfaction  통과
wt3 평균 1.0000 (기대 1.0000), 최대 20.7537 (기대 20.7537)


> **검증 통과** <br>
> 세 변수의 빈도와 가중치 분포가 코드북 기재값과 일치. 원본 무결성 확인됨.

### 1.3 분석 모집단 확정 및 변수 축소
- 이 조사는 종사상지위(emp_type)에 따라 문항이 분기됨
- 임금근로자(emp_type=3)로 한정하면 공통 문항에 더해 Q11~Q26(고용형태, 상사 자질, 사업장 평가, 노조 유무)을 사용할 수 있음
- 판단이 필요 없는 변수부터 기계적으로 제거: 주관식 기타(_etc), 동거 가구원 정보(hm_), 타 지위 전용 문항(selfemp_/semp_/unfw_), 결측률 90% 초과 문항
- 설계 정보(id, wt1~wt3, stratification, district, household)는 복합표본 분석에 필요하므로 보존

### 1.4 결측코드 처리
- 무응답 유형별 코드: 7/77/777 = 해당없음, 8/88/888 = 모름·무응답, 9/99/999 = 거절
- 주의: 코드의 자릿수가 변수마다 다름. 유효값 범위와 겹치지 않도록 설계되어 있음
- 일괄 치환 금지. age의 77·88은 실제 나이(383명), ctime의 7·8·9는 실제 통근시간(분, 559명)
- 검산 기준: 전체 표본에서 ctime의 777/888/999를 제거하면 코드북 기재값(유효 47,335 / 평균 38.17 / 표준편차 32.361)과 일치해야 함

In [6]:
import pandas as pd
import numpy as np

# --- 01-1. 모집단 한정 ---
emp = df[df['emp_type'] == '3'].copy()
print(f'임금근로자 : {len(emp):,}명')

# --- 01-2. 기계적 축소 ---
drop_cols = (
    [c for c in emp.columns if c.endswith('_etc')]
  + [c for c in emp.columns if c.startswith('hm_')]
  + [c for c in emp.columns if c.startswith(('selfemp_', 'semp_', 'unfw_'))]
)
emp = emp.drop(columns=drop_cols)

# 임금근로자에게 전원 결측이거나 결측률 90% 넘는 열 제거
na_ratio = emp.isna().mean()
emp = emp.drop(columns=na_ratio[na_ratio > 0.90].index)
print(f'변수 : 439 → {emp.shape[1]}개')

# --- 01-3. 결측코드 처리 시범 (3개만) ---
emp['satisfaction'] = pd.to_numeric(emp['satisfaction'], errors='coerce')
emp.loc[emp['satisfaction'].isin([8, 9]), 'satisfaction'] = np.nan

emp['ctime'] = pd.to_numeric(emp['ctime'], errors='coerce')
emp.loc[emp['ctime'].isin([777, 888, 999]), 'ctime'] = np.nan

emp['age'] = pd.to_numeric(emp['age'], errors='coerce')   # 결측코드 없음, 그대로

print(f"\nsatisfaction 유효 : {emp['satisfaction'].notna().sum():,}")
print(f"ctime  평균 {emp['ctime'].mean():.2f} / 표준편차 {emp['ctime'].std():.3f}")
print(f"age    평균 {emp['age'].mean():.2f} / 최대 {emp['age'].max():.0f}")

임금근로자 : 30,150명
변수 : 439 → 291개

satisfaction 유효 : 30,012
ctime  평균 43.33 / 표준편차 29.674
age    평균 48.23 / 최대 93


> **가중치 재표준화 필요** <br>
> wt3는 전체 50,195명 기준으로 평균 1.0이 되도록 표준화된 값. 임금근로자만 추출하면 평균이 1.2802로 틀어지므로 부분집합 기준으로 재표준화해야 함.